# ⚽💰 Football Finance Intelligence
## Module 3 — Rentabilité des transferts et efficacité des clubs

**Idée :** les modules précédents ont étudié le marché et la valeur des joueurs. Ici, on se place du côté des **clubs** : leurs achats sont-ils rentables, et leur argent se transforme-t-il en résultats sportifs ?

### Questions étudiées
1. **ROI de revente :** quels transferts ont rapporté le plus (ou fait perdre le plus) entre l'achat et la revente ?
2. **Les clubs « traders » :** quels clubs réalisent les plus grosses plus-values sur leurs ventes ?
3. **Coût de la performance :** combien coûte un but, ou une minute jouée, selon le joueur ?
4. **Moneyball :** quels clubs obtiennent **plus de points que ne le prédit la valeur de leur effectif** ?

In [1]:
!pip install duckdb -q
!curl -sLO https://pub-e682421888d945d684bcae8890b0ec20.r2.dev/data/transfermarkt-datasets.duckdb

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

con = duckdb.connect("transfermarkt-datasets.duckdb", read_only=True)
def q(sql):
    return con.sql(sql).df()

# Structure des tables utiles pour ce module
for table in ["clubs", "club_games", "games", "appearances"]:
    print(f"\n=== {table} ===")
    print(q(f"DESCRIBE {table}")["column_name"].tolist())


=== clubs ===
['club_id', 'club_code', 'name', 'domestic_competition_id', 'total_market_value', 'squad_size', 'average_age', 'foreigners_number', 'foreigners_percentage', 'national_team_players', 'stadium_name', 'stadium_seats', 'net_transfer_record', 'coach_name', 'last_season', 'filename', 'url']

=== club_games ===
['game_id', 'club_id', 'own_goals', 'own_position', 'own_manager_name', 'opponent_id', 'opponent_goals', 'opponent_position', 'opponent_manager_name', 'hosting', 'is_win']

=== games ===
['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'home_club_position', 'away_club_position', 'home_club_manager_name', 'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url', 'home_club_formation', 'away_club_formation', 'home_club_name', 'away_club_name', 'aggregate', 'competition_type']

=== appearances ===
['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'dat

### Tables utilisées dans ce module
| Table | Contenu utile |
|---|---|
| `transfers` | Prix d'achat et de revente des joueurs |
| `clubs` | Valeur totale de l'effectif (`total_market_value`) |
| `club_games` | Résultat de chaque match pour chaque club (buts, victoire) |
| `games` | Saison et compétition de chaque match |
| `appearances` | Minutes, buts et passes décisives de chaque joueur par match |

## 1. Le ROI de revente : acheter, puis revendre
**Méthode :** on reconstitue chaque **cycle achat → revente** : un club achète un joueur avec indemnité, puis le revend avec indemnité. Achats depuis juillet 2012, détention d'au moins 6 mois.

**Indicateurs :**
- **ROI** = prix de vente ÷ prix d'achat − 1
- **Rendement annualisé** = (prix de vente ÷ prix d'achat)^(1 / durée) − 1, pour comparer des durées de détention différentes
- **Plus-value comptable** = prix de vente − valeur nette comptable (amortissement linéaire sur 5 ans, comme au Module 2)

La différence entre les deux plus-values est importante : un joueur revendu **moins cher** qu'il n'a été acheté peut quand même générer une **plus-value comptable**, car sa valeur dans les comptes a été amortie entre-temps.

*Limite : seuls les joueurs revendus avec indemnité sont inclus. Les joueurs partis libres (souvent des échecs) sont exclus, ce qui rend les résultats plus favorables que la réalité.*

In [3]:
cycles = q("""
WITH achats AS (
    SELECT t.player_id, t.player_name,
           t.to_club_id   AS club_id,
           t.to_club_name AS club,
           t.transfer_date AS date_achat,
           CAST(t.transfer_fee AS DOUBLE) AS prix_achat,
           YEAR(t.transfer_date) - YEAR(TRY_CAST(p.date_of_birth AS DATE)) AS age_achat
    FROM transfers t
    JOIN players p ON t.player_id = p.player_id
    WHERE t.transfer_fee > 0
      AND t.transfer_date BETWEEN DATE '2012-07-01' AND CURRENT_DATE
),
ventes AS (
    SELECT player_id,
           from_club_id  AS club_id,
           transfer_date AS date_vente,
           CAST(transfer_fee AS DOUBLE) AS prix_vente,
           to_club_name  AS acheteur
    FROM transfers
    WHERE transfer_fee > 0
      AND transfer_date <= CURRENT_DATE
),
cycles AS (
    SELECT a.*, v.date_vente, v.prix_vente, v.acheteur,
           ROW_NUMBER() OVER (PARTITION BY a.player_id, a.club_id, a.date_achat
                              ORDER BY v.date_vente) AS rn
    FROM achats a
    JOIN ventes v ON a.player_id = v.player_id
                 AND a.club_id   = v.club_id
                 AND v.date_vente > a.date_achat
)
SELECT * EXCLUDE (rn) FROM cycles WHERE rn = 1
""")

# --- Calculs ---
cycles["duree_ans"] = (pd.to_datetime(cycles["date_vente"])
                       - pd.to_datetime(cycles["date_achat"])).dt.days / 365.25
cycles = cycles[cycles["duree_ans"] >= 0.5].copy()
cycles["age_achat"] = cycles["age_achat"].astype(float)
cycles["prix_achat_M"] = cycles["prix_achat"] / 1e6
cycles["prix_vente_M"] = cycles["prix_vente"] / 1e6
cycles["roi"] = cycles["prix_vente"] / cycles["prix_achat"] - 1
cycles["roi_annuel"] = (cycles["prix_vente"] / cycles["prix_achat"]) ** (1 / cycles["duree_ans"]) - 1
cycles["plus_value_M"] = cycles["prix_vente_M"] - cycles["prix_achat_M"]
vnc = cycles["prix_achat_M"] * (1 - cycles["duree_ans"].clip(upper=5) / 5)
cycles["pv_comptable_M"] = cycles["prix_vente_M"] - vnc

print(f"Nombre de cycles achat → revente : {len(cycles)}")
print(f"Revendus plus cher qu'achetés   : {(cycles['roi'] > 0).mean():.1%}")
print(f"En plus-value comptable         : {(cycles['pv_comptable_M'] > 0).mean():.1%}")
print(f"ROI médian : {cycles['roi'].median():+.1%} | rendement annualisé médian : "
      f"{cycles['roi_annuel'].median():+.1%} | durée médiane : {cycles['duree_ans'].median():.1f} ans")

# --- Graphique 1 : prix d'achat vs prix de vente ---
cycles["resultat"] = np.where(cycles["roi"] > 0, "Revendu plus cher", "Revendu moins cher")
fig = px.scatter(cycles, x="prix_achat_M", y="prix_vente_M", color="resultat",
                 log_x=True, log_y=True, opacity=0.6, hover_name="player_name",
                 hover_data={"club": True, "acheteur": True, "age_achat": True,
                             "duree_ans": ":.1f", "resultat": False},
                 color_discrete_map={"Revendu plus cher": "#00CC96",
                                     "Revendu moins cher": "#EF553B"},
                 title="Prix d'achat vs prix de revente (échelle logarithmique, M€)",
                 labels={"prix_achat_M": "Prix d'achat (M€)",
                         "prix_vente_M": "Prix de revente (M€)", "resultat": ""})
fig.add_trace(go.Scatter(x=[0.05, 250], y=[0.05, 250], mode="lines",
                         line=dict(dash="dash", color="black"),
                         name="Revendu au même prix"))
fig.show()

# --- Graphique 2 : par âge à l'achat ---
cycles["tranche"] = pd.cut(cycles["age_achat"], bins=[0, 21, 24, 27, 45],
                           labels=["21 ans et -", "22-24", "25-27", "28 et +"])
par_age = (cycles.groupby("tranche", observed=True)
           .agg(nb=("roi", "count"),
                roi_median=("roi", lambda s: s.median() * 100),
                pct_revendu_plus_cher=("roi", lambda s: (s > 0).mean() * 100),
                pct_pv_comptable=("pv_comptable_M", lambda s: (s > 0).mean() * 100))
           .reset_index())

fig = go.Figure([
    go.Bar(x=par_age["tranche"], y=par_age["pct_revendu_plus_cher"],
           name="% revendus plus cher qu'achetés"),
    go.Bar(x=par_age["tranche"], y=par_age["pct_pv_comptable"],
           name="% en plus-value comptable"),
])
fig.update_layout(barmode="group", title="Rentabilité de la revente selon l'âge à l'achat",
                  xaxis_title="Âge à l'achat", yaxis_title="%")
fig.show()
print(par_age.round(1).to_string(index=False))

# --- Tops et flops ---
cols = ["player_name", "club", "acheteur", "age_achat",
        "prix_achat_M", "prix_vente_M", "duree_ans", "plus_value_M"]
print("\n🟢 Top 10 des plus-values (M€) :")
print(cycles.nlargest(10, "plus_value_M")[cols].round(1).to_string(index=False))
print("\n🔴 Top 10 des moins-values (M€) :")
print(cycles.nsmallest(10, "plus_value_M")[cols].round(1).to_string(index=False))

Nombre de cycles achat → revente : 5738
Revendus plus cher qu'achetés   : 61.7%
En plus-value comptable         : 84.2%
ROI médian : +50.0% | rendement annualisé médian : +19.5% | durée médiane : 2.0 ans


    tranche   nb  roi_median  pct_revendu_plus_cher  pct_pv_comptable
21 ans et - 1767       120.0                   73.0              92.3
      22-24 2225        60.9                   64.2              86.7
      25-27 1286         5.1                   51.5              78.6
    28 et +  459       -30.0                   34.0              57.1

🟢 Top 10 des plus-values (M€) :
      player_name        club    acheteur  age_achat  prix_achat_M  prix_vente_M  duree_ans  plus_value_M
           Neymar   Barcelona         PSG       21.0          88.0         222.0        4.1         134.0
Philippe Coutinho   Liverpool   Barcelona       21.0          13.0         135.0        4.9         122.0
  Ousmane Dembélé    Dortmund   Barcelona       19.0          35.0         148.0        1.1         113.0
  Jude Bellingham    Dortmund Real Madrid       17.0          30.2         127.0        2.9          96.8
   Moisés Caicedo    Brighton     Chelsea       20.0          28.2         116.0       

### Lecture
**Vue d'ensemble :** 5 738 cycles achat → revente. ROI médian **+50 %**, détention médiane **2 ans**, soit environ **+19,5 % par an**.

| Âge à l'achat | Nb | ROI médian | Revendus plus cher | Plus-value comptable |
|---|---|---|---|---|
| 21 ans et moins | 1 767 | **+120 %** | 73 % | 92 % |
| 22-24 ans | 2 225 | +61 % | 64 % | 87 % |
| 25-27 ans | 1 286 | +5 % | 52 % | 79 % |
| 28 ans et plus | 459 | **−30 %** | 34 % | 57 % |

**1. L'âge reste le facteur clé**, en cohérence avec le Module 2 : plus le joueur est acheté jeune, plus la revente est rentable.

**2. Gain économique ≠ gain comptable.** 62 % des joueurs sont revendus plus cher qu'achetés, mais **84 %** des ventes génèrent une **plus-value comptable**, car l'amortissement a réduit leur valeur dans les comptes. 💼 Un club peut donc afficher des plus-values comptables tout en perdant de l'argent sur ses joueurs.

**3. Un problème de liquidité pour les achats très chers.** En dessous de 5 M€, la plupart des joueurs sont revendus plus cher. Au-delà de 50 M€, les pertes dominent : très peu de clubs peuvent racheter un joueur à ce prix.

**4. La malédiction du vainqueur.** Neymar, Coutinho, Dembélé et Antony figurent à la fois parmi les plus grosses plus-values et les plus grosses moins-values. Le vendeur gagne, et l'acheteur qui a payé au plus haut perd presque autant à la revente. C'est le *winner's curse* des enchères : celui qui l'emporte est souvent celui qui a le plus surestimé l'actif.

**5. Le cas du FC Barcelone.** Acheteur dans 2 des 10 plus grosses plus-values (Coutinho, Dembélé) et vendeur dans 4 des 10 plus grosses moins-values (Coutinho, Dembélé, Griezmann, Suárez) : le club a acheté au sommet du marché et revendu à perte.

*Limite : seuls les joueurs revendus avec indemnité sont inclus. Les départs libres sont exclus, ce qui rend ces résultats plus favorables que la réalité.*

## 2. Les clubs « traders » : qui gagne de l'argent sur ses reventes ?
**Méthode :** pour chaque club, on additionne ses cycles achat → revente :
- **Investi** = somme des prix d'achat
- **Encaissé** = somme des prix de revente
- **Plus-value totale** = encaissé − investi
- **ROI global** = encaissé ÷ investi − 1

On retient les clubs ayant réalisé au moins **15 cycles** depuis 2012.

In [4]:
clubs_trading = (cycles.groupby("club_id")
    .agg(club=("club", "first"),
         nb_cycles=("roi", "count"),
         investi_M=("prix_achat_M", "sum"),
         encaisse_M=("prix_vente_M", "sum"),
         plus_value_M=("plus_value_M", "sum"),
         pv_comptable_M=("pv_comptable_M", "sum"),
         roi_median=("roi", lambda s: s.median() * 100),
         pct_gagnants=("roi", lambda s: (s > 0).mean() * 100))
    .reset_index())
clubs_trading["roi_global"] = (clubs_trading["encaisse_M"] / clubs_trading["investi_M"] - 1) * 100

ct = clubs_trading[clubs_trading["nb_cycles"] >= 15]
top  = ct.nlargest(15, "plus_value_M")
flop = ct.nsmallest(10, "plus_value_M")
both = pd.concat([top, flop]).drop_duplicates("club_id").sort_values("plus_value_M")

fig = px.bar(both, x="plus_value_M", y="club", orientation="h",
             color="roi_global", color_continuous_scale="RdYlGn",
             color_continuous_midpoint=0,
             hover_data={"nb_cycles": True, "investi_M": ":.0f",
                         "encaisse_M": ":.0f", "pct_gagnants": ":.0f"},
             title="Plus-value totale réalisée à la revente : top 15 et flop 10 des clubs (M€)",
             labels={"plus_value_M": "Plus-value totale (M€)", "club": "",
                     "roi_global": "ROI global (%)"})
fig.update_layout(height=750)
fig.show()

cols = ["club", "nb_cycles", "investi_M", "encaisse_M", "plus_value_M",
        "roi_global", "pct_gagnants"]
print("🟢 Top 15 des clubs traders :")
print(top[cols].round(0).to_string(index=False))
print("\n🔴 Flop 10 :")
print(flop[cols].round(0).to_string(index=False))

🟢 Top 15 des clubs traders :
       club  nb_cycles  investi_M  encaisse_M  plus_value_M  roi_global  pct_gagnants
   Dortmund         31      465.0       814.0         349.0        75.0          55.0
      Lille         35      235.0       580.0         346.0       147.0          74.0
     Monaco         43      554.0       886.0         332.0        60.0          47.0
Atalanta BC         40      351.0       642.0         291.0        83.0          60.0
    Leipzig         35      571.0       854.0         283.0        50.0          57.0
  Brentford         17       67.0       340.0         273.0       407.0          76.0
   Salzburg         38      144.0       415.0         271.0       188.0          68.0
      Porto         30      250.0       513.0         263.0       105.0          57.0
    Benfica         48      494.0       744.0         250.0        51.0          48.0
       Ajax         34      340.0       583.0         243.0        71.0          41.0
   Sporting         27   

### Lecture : deux mondes du football
**Les clubs vendeurs** font de la revente un modèle économique :
- **Dortmund** (+349 M€), **Lille** (+346 M€) et **Monaco** (+332 M€) forment le podium.
- Les « écoles de vente » des championnats formateurs (**Porto, Benfica, Ajax, Sporting**) sont toutes présentes.
- La **Ligue 1** place 2 clubs dans le top 3, ce qui confirme son rôle d'exportatrice (Module 1).
- Le réseau **Red Bull** (Salzburg + Leipzig) cumule environ **+554 M€** grâce à un modèle multi-clubs.
- **Brentford** est le plus efficace : 67 M€ investis → 340 M€ encaissés (**ROI +407 %**). Comme **Brighton**, il est réputé pour son recrutement fondé sur la data.

**Les clubs acheteurs** perdent à la revente :
- **Chelsea** (−389 M€ sur 1,16 Md€ investis) et **Manchester United** (−333 M€, seulement **6 %** de cycles gagnants).
- PSG, Barcelone, Arsenal, Juventus, Bayern et Manchester City sont également en perte.

**Les deux mondes sont liés :** les clubs acheteurs paient au prix fort les joueurs formés ou valorisés par les clubs vendeurs. La plus-value des uns est la moins-value des autres.

⚠️ **Perdre à la revente n'est pas forcément un échec :** les grands clubs achètent pour **gagner**, pas pour revendre. La moins-value est alors le coût de la performance sportive. Il faut donc mesurer si l'argent se transforme en **points** : c'est l'objet de la section suivante.

## 3. Moneyball : l'argent se transforme-t-il en points ?
**Méthode :**
1. **Points** de chaque club par saison en championnat (Big 5, saisons 2015/16 à 2025/26) : victoire = 3, nul = 1. On utilise les **points par match** (la Bundesliga joue 34 matchs, les autres 38).
2. **Valeur de l'effectif** = somme des valeurs marchandes, au 1ᵉʳ août, des joueurs ayant joué au moins 450 minutes en championnat dans la saison.
3. **Valeur relative** = valeur de l'effectif ÷ moyenne de sa ligue la même saison. Cela permet de comparer les ligues et les époques.
4. **Régression** : points par match = a + b × ln(valeur relative). Le logarithme traduit le fait que doubler son budget n'apporte pas deux fois plus de points.
5. **Surperformance** = (points réels − points prédits), ramenée à une saison de 38 matchs. Positive : le club fait mieux que ne le permet son budget.

Le **R²** indique la part des écarts de points expliquée par l'argent.

In [5]:
big5_codes = "('GB1','ES1','IT1','L1','FR1')"

mb = q(f"""
WITH pts AS (
    SELECT cg.club_id, g.competition_id AS ligue,
           TRY_CAST(g.season AS INT) AS saison,
           COUNT(*) AS matchs,
           SUM(CASE WHEN cg.own_goals > cg.opponent_goals THEN 3
                    WHEN cg.own_goals = cg.opponent_goals THEN 1
                    ELSE 0 END) AS points
    FROM club_games cg
    JOIN games g ON cg.game_id = g.game_id
    WHERE g.competition_id IN {big5_codes}
    GROUP BY 1, 2, 3
),
effectif AS (
    SELECT a.player_club_id AS club_id,
           TRY_CAST(g.season AS INT) AS saison,
           a.player_id,
           SUM(a.minutes_played) AS minutes,
           ANY_VALUE(MAKE_DATE(TRY_CAST(g.season AS INT), 8, 1)) AS date_ref
    FROM appearances a
    JOIN games g ON a.game_id = g.game_id
    WHERE g.competition_id IN {big5_codes}
    GROUP BY 1, 2, 3
    HAVING SUM(a.minutes_played) >= 450
),
valo AS (
    SELECT e.club_id, e.saison, e.player_id, v.market_value_in_eur AS valeur
    FROM effectif e
    ASOF LEFT JOIN player_valuations v
      ON e.player_id = v.player_id AND e.date_ref >= v.date
),
val_club AS (
    SELECT club_id, saison,
           SUM(valeur) / 1e6 AS valeur_effectif_M,
           COUNT(*) AS nb_joueurs
    FROM valo
    GROUP BY 1, 2
)
SELECT p.*, c.name AS club, vc.valeur_effectif_M, vc.nb_joueurs
FROM pts p
JOIN val_club vc ON p.club_id = vc.club_id AND p.saison = vc.saison
LEFT JOIN clubs c ON p.club_id = c.club_id
WHERE p.saison BETWEEN 2015 AND 2025
""")

# --- Préparation ---
mb = mb[(mb["matchs"] >= 30) & (mb["valeur_effectif_M"] > 0)].copy()
noms_ligues = {"GB1": "Premier League", "ES1": "LaLiga", "IT1": "Serie A",
               "L1": "Bundesliga", "FR1": "Ligue 1"}
mb["ligue"] = mb["ligue"].map(noms_ligues)
mb["ppm"] = mb["points"] / mb["matchs"]
mb["valeur_rel"] = mb["valeur_effectif_M"] / mb.groupby(["ligue", "saison"])["valeur_effectif_M"].transform("mean")
mb["log_val"] = np.log(mb["valeur_rel"])

# --- Régression ---
b, a = np.polyfit(mb["log_val"], mb["ppm"], 1)
r2 = np.corrcoef(mb["log_val"], mb["ppm"])[0, 1] ** 2
mb["ppm_predit"] = a + b * mb["log_val"]
mb["surperf_pts"] = (mb["ppm"] - mb["ppm_predit"]) * 38
mb["saison_txt"] = mb["saison"].astype(int).astype(str)

print(f"{len(mb)} saisons-clubs analysées")
print(f"R² = {r2:.2f} → l'argent explique {r2:.0%} des écarts de points")
print(f"Doubler sa valeur relative rapporte environ {b * np.log(2) * 38:.1f} points sur une saison")

# --- Graphique 1 : la droite du Moneyball ---
fig = px.scatter(mb, x="valeur_rel", y="ppm", color="ligue", log_x=True, opacity=0.6,
                 hover_name="club", hover_data={"saison_txt": True, "points": True,
                 "valeur_effectif_M": ":.0f", "surperf_pts": ":.1f",
                 "valeur_rel": ":.2f", "ligue": False},
                 title=f"Moneyball : valeur de l'effectif vs points par match (R² = {r2:.2f})",
                 labels={"valeur_rel": "Valeur relative de l'effectif (1 = moyenne de la ligue)",
                         "ppm": "Points par match", "ligue": "Ligue", "saison_txt": "Saison"})
x_line = np.linspace(mb["valeur_rel"].min(), mb["valeur_rel"].max(), 100)
fig.add_trace(go.Scatter(x=x_line, y=a + b * np.log(x_line), mode="lines",
                         line=dict(color="black", dash="dash"), name="Points prédits par le budget"))
fig.show()

# --- Graphique 2 : les clubs les plus et les moins efficaces ---
eff = (mb.groupby("club")
         .agg(nb_saisons=("saison", "count"),
              surperf_moy=("surperf_pts", "mean"),
              valeur_rel_moy=("valeur_rel", "mean"))
         .reset_index())
eff = eff[eff["nb_saisons"] >= 5]
classement = pd.concat([eff.nlargest(12, "surperf_moy"),
                        eff.nsmallest(8, "surperf_moy")]).sort_values("surperf_moy")

fig = px.bar(classement, x="surperf_moy", y="club", orientation="h",
             color="surperf_moy", color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
             hover_data={"nb_saisons": True, "valeur_rel_moy": ":.2f"},
             title="Efficacité Moneyball : points gagnés au-delà du budget (moyenne par saison)",
             labels={"surperf_moy": "Points au-dessus / en dessous de la prédiction", "club": ""})
fig.update_layout(height=650, coloraxis_showscale=False)
fig.show()

print(classement.sort_values("surperf_moy", ascending=False).round(2).to_string(index=False))

1052 saisons-clubs analysées
R² = 0.57 → l'argent explique 57% des écarts de points
Doubler sa valeur relative rapporte environ 11.1 points sur une saison


                   club  nb_saisons  surperf_moy  valeur_rel_moy
                RC Lens           6        17.24            0.63
        Manchester City          11        12.63            2.43
           Liverpool FC          11        10.47            1.88
      1.FC Union Berlin           7         9.65            0.44
          Bayern Munich          11         9.47            4.06
               SD Eibar           6         8.81            0.26
             SSC Napoli          11         8.40            2.08
           FC Barcelona          11         8.18            3.61
            Atalanta BC          11         7.73            1.13
       Deportivo Alavés           9         7.08            0.27
      Stade Brestois 29           6         6.61            0.42
            Inter Milan          11         6.30            2.43
           UC Sampdoria           8        -6.18            0.61
Wolverhampton Wanderers           8        -6.46            0.70
             Everton FC  

### Lecture
**1. L'argent explique 57 % des points (R² = 0,57).** Les 43 % restants relèvent du coaching, de la tactique, de la qualité du recrutement et de la chance. Doubler sa valeur relative rapporte environ **11 points par saison**, avec des rendements décroissants.

**2. Les clubs les plus efficaces avec peu de moyens**
| Club | Surperformance (pts/saison) | Valeur relative | Fait marquant |
|---|---|---|---|
| RC Lens | **+17,2** | 0,63 | 2ᵉ de Ligue 1 en 2022/23 |
| Union Berlin | +9,7 | 0,44 | Qualifié en Ligue des champions |
| Eibar | +8,8 | 0,26 | Maintien en LaLiga avec l'un des plus petits effectifs |
| Stade Brestois | +6,6 | 0,42 | Qualifié en Ligue des champions |

**3. Le paradoxe des géants.** Manchester City (+12,6), Bayern (+9,5) et Barcelone (+8,2) perdent de l'argent à la revente, mais transforment très bien leur budget en points.

⚠️ *Limite : le modèle sous-estime les clubs très dominants (valeur relative > 3), presque tous au-dessus de la droite. Un club bien plus riche que ses rivaux bénéficie d'un avantage supérieur à ce que prévoit le modèle (effet « le gagnant rafle tout »). Leur surperformance est donc en partie un effet du modèle.*

**4. Les clubs les moins efficaces.** **Schalke 04** (−11,9 points par saison avec un budget dans la moyenne, relégué en 2021), **Southampton** (−10,8), **Wolfsburg** (−9,3) et **Valencia** (−8,6).

**5. Atalanta, la double efficacité :** dans le top des clubs traders (+291 M€) **et** du Moneyball (+7,7 points par saison).

## 4. La matrice finale : rentabilité financière × efficacité sportive
On croise les deux mesures du module pour chaque club :
- **Axe horizontal :** ROI global des reventes (section 2)
- **Axe vertical :** surperformance Moneyball (section 3)

| | Perd à la revente | Gagne à la revente |
|---|---|---|
| **Plus de points que prévu** | 🏆 Champions qui dépensent | ⭐ Modèle complet |
| **Moins de points que prévu** | 💸 Gaspilleurs | 🏪 Vendeurs |

In [6]:
# Efficacité Moneyball par club_id (pour pouvoir croiser avec les reventes)
eff_id = (mb.groupby("club_id")
            .agg(club=("club", "first"),
                 nb_saisons=("saison", "count"),
                 surperf_moy=("surperf_pts", "mean"))
            .reset_index())

matrice = eff_id.merge(clubs_trading[["club_id", "nb_cycles", "roi_global", "plus_value_M"]],
                       on="club_id", how="inner")
matrice = matrice[(matrice["nb_saisons"] >= 5) & (matrice["nb_cycles"] >= 10)].copy()

def profil(r):
    if r["surperf_moy"] >= 0 and r["roi_global"] >= 0: return "⭐ Modèle complet"
    if r["surperf_moy"] >= 0:                          return "🏆 Champions qui dépensent"
    if r["roi_global"] >= 0:                           return "🏪 Vendeurs"
    return "💸 Gaspilleurs"
matrice["profil"] = matrice.apply(profil, axis=1)

# On affiche le nom seulement pour les clubs les plus marquants
matrice["etiquette"] = np.where((matrice["surperf_moy"].abs() > 5) |
                                (matrice["roi_global"] > 150) |
                                (matrice["roi_global"] < -25),
                                matrice["club"], "")

fig = px.scatter(matrice, x="roi_global", y="surperf_moy", color="profil",
                 size="nb_cycles", text="etiquette", hover_name="club",
                 hover_data={"plus_value_M": ":.0f", "nb_saisons": True,
                             "etiquette": False, "profil": False},
                 title="Matrice des clubs : rentabilité des reventes × efficacité sportive",
                 labels={"roi_global": "ROI global des reventes (%)",
                         "surperf_moy": "Surperformance Moneyball (points/saison)",
                         "profil": "Profil"})
fig.update_traces(textposition="top center", textfont_size=10)
fig.add_vline(x=0, line_dash="dash", line_color="black")
fig.add_hline(y=0, line_dash="dash", line_color="black")
fig.update_layout(height=700)
fig.show()

for p in ["⭐ Modèle complet", "🏆 Champions qui dépensent", "🏪 Vendeurs", "💸 Gaspilleurs"]:
    clubs_p = matrice[matrice["profil"] == p].sort_values("surperf_moy", ascending=False)
    print(f"\n{p} ({len(clubs_p)} clubs) : {', '.join(clubs_p['club'].head(10))}")


⭐ Modèle complet (33 clubs) : RC Lens, Liverpool FC, 1.FC Union Berlin, SSC Napoli, Atalanta BC, Stade Brestois 29, Inter Milan, SC Freiburg, LOSC Lille, RB Leipzig

🏆 Champions qui dépensent (8 clubs) : Manchester City, Bayern Munich, FC Barcelona, Arsenal FC, Juventus FC, Paris Saint-Germain, Real Betis Balompié, AC Milan

🏪 Vendeurs (36 clubs) : Sevilla FC, FC Lorient, Celta de Vigo, FC Toulouse, Montpellier HSC, Real Valladolid CF, Stade Rennais FC, US Lecce, Olympique Marseille, FC Empoli

💸 Gaspilleurs (8 clubs) : Manchester United, Aston Villa, Tottenham Hotspur, Chelsea FC, Hertha BSC, West Ham United, Valencia CF, FC Schalke 04


### Lecture
| Profil | Nb | Exemples | Interprétation |
|---|---|---|---|
| ⭐ Modèle complet | 33 | Lens, Liverpool, Union Berlin, Napoli, Atalanta, Brest, Inter, Freiburg, Lille, Leipzig, Brentford | Gagnent de l'argent **et** des points |
| 🏆 Champions qui dépensent | 8 | Manchester City, Bayern, Barcelone, Arsenal, Juventus, PSG, Betis, AC Milan | Paient le prix de la performance |
| 🏪 Vendeurs | 36 | Séville, Toulouse, Montpellier, Rennes, Marseille, Lecce, Celta | Se financent par les ventes, au détriment des résultats |
| 💸 Gaspilleurs | 8 | Manchester United, Aston Villa, Tottenham, Chelsea, West Ham, Hertha, Valencia, Schalke | Perdent sur les deux tableaux |

- **Liverpool** et l'**Inter** sont les seuls grands clubs dans le « modèle complet » : ils savent vendre cher et réinvestir efficacement.
- **5 des 8 « gaspilleurs » sont anglais.** Hypothèse : l'abondance de ressources en Premier League (Module 1) réduit la pression à l'efficacité.
- **Lecce** est un cas extrême de club vendeur (ROI d'environ +580 %).

*Limite : seules les reventes avec indemnité sont comptées, donc l'axe horizontal est décalé vers la droite (69 clubs sur 85 ont un ROI positif). La comparaison entre clubs reste valable.*

---
# ✅ Conclusion du Module 3 — Rentabilité des transferts et efficacité des clubs

### Les 5 messages clés
1. **La revente est rentable… pour les jeunes.** ROI médian de +120 % pour les joueurs achetés à 21 ans ou moins, contre −30 % à 28 ans et plus.
2. **Gain comptable ≠ gain économique.** 84 % des reventes génèrent une plus-value comptable, alors que seulement 62 % des joueurs sont revendus plus cher qu'achetés : l'amortissement embellit les comptes.
3. **La malédiction du vainqueur.** Les plus grosses plus-values des uns (Neymar, Coutinho, Dembélé) deviennent les plus grosses moins-values des autres. L'acheteur au prix fort perd presque toujours.
4. **L'argent explique 57 % des points.** Doubler la valeur de son effectif rapporte environ 11 points par saison. Les 43 % restants laissent une large place à l'efficacité : Lens, Union Berlin et Brest font bien mieux que leur budget.
5. **Quatre profils de clubs.** Le « modèle complet » (Lens, Liverpool, Atalanta, Brentford) prouve qu'on peut gagner à la fois de l'argent et des points. Les « gaspilleurs » (Manchester United, Chelsea, Tottenham) perdent sur les deux tableaux.

### Limites
- Seuls les joueurs revendus avec indemnité sont inclus dans les cycles achat → revente.
- Le modèle Moneyball sous-estime les clubs très dominants.
- Contrat de 5 ans supposé pour le calcul de la valeur comptable.

➡️ **Module suivant :** un modèle de machine learning pour estimer la « juste valeur » d'un joueur et repérer les joueurs sous-évalués.